# Pre-registration: when should you not trust an attribution graph?

**Author:** Jason Soo
**Written:** [16-09-2026 6:17pm] — before any corpus data is collected
**Status:** append-only. Do not edit after timestamping; add a dated entry to the Deviations log instead.

---

## 0. What I already know, and how

Honesty requires separating what I am predicting from what I have already seen. Day 1 involved a
10-prompt pilot, so some predictions below are **data-informed**, not blind. Flagged where so.

Established on Day 1, on `google/gemma-2-2b` with GemmaScope per-layer transcoders via
`circuit-tracer`:

| Finding | Value |
|---|---|
| Replacement score, France prompt | 0.727 (library's own: 0.7196) |
| Completeness, France prompt | 0.936 (library's: 0.9268) |
| Run-to-run noise floor (bf16) | ~5e-05 on replacement score |
| Structure reproducibility | `n_nodes` identical across runs |
| Series termination | ~21 hops, confirming DAG structure |
| Node layout | errors then embeddings, contiguous; error grid is layer-major (confirmed by mod-6 zero pattern) |
| `max_feature_nodes=8192` artefact | saturation injected spurious error mass at the final token position; resolved at 16384 |
| Pilot replacement range (n=10) | 0.684 – 0.755 |
| Pilot completeness range (n=10) | 0.925 – 0.942 |

The ~0.008 discrepancy between my implementation and the library's is 150x the noise floor and
therefore a genuine methodological difference, not rounding. I use the library's implementation as
primary and mine as a cross-check. To be resolved by reading `METRIC_FN`'s source; recorded either
way.

---

## 1. Question

Anthropic report that attribution graphs gave satisfying insight on roughly a quarter of prompts
tried, and state this is difficult to quantify precisely — it is a self-report over an unspecified
prompt set, not a measurement. Each attempt costs hours of expert time. No pre-flight check exists.

The library computes replacement score and completeness, and the methods paper notes these are
useful "to flag prompts our method performs poorly on" — then never develops that use. The metrics
are reported as dataset-wide averages for comparing dictionaries, never as a per-prompt triage
instrument, and nobody has tested whether they predict a human's success at reading a graph.

**Primary question.** Across a stratified prompt corpus, do cheap prompt-side properties predict
attribution-graph quality well enough to serve as a triage rule?

**Secondary question.** Does replacement score track whether a human can actually extract a
mechanism from the graph?

---

## 2. Design

**Model:** `google/gemma-2-2b`, bf16, GemmaScope per-layer transcoders, nnsight backend,
`max_feature_nodes=16384`, `batch_size=32`. Full config in `day1_config.json`.

**Corpus:** ~300 prompts, 9 categories, ~33 each. Every category is grounded in a limitation the
methods paper documents qualitatively but never quantifies.

| # | Category | Grounded in |
|---|---|---|
| 1 | Factual recall | published success case |
| 2 | Two-digit arithmetic | published success case |
| 3 | Syntactic agreement | short mechanical control |
| 4 | Multi-hop | longer causal chains, error accumulation |
| 5 | Code completion | untested territory |
| 6 | Induction / copying | attention blind spot (frozen QK) |
| 7 | Multiple choice | attention blind spot |
| 8 | Known vs unknown entity | inactive-feature / suppression blind spot |
| 9 | Obfuscated text (4 corruption rungs) | reconstruction-error blind spot |

**Controls.** Prompt length varied deliberately *within* each category (short / medium / long) so
length is not confounded with category. Hard ceiling at `MAX_TOKENS`, applied identically to all
categories. Every prompt filtered to top-1 confidence in a specified band, so no category is
dominated by trivially certain or near-random continuations.

**Saturation guard.** Any graph reaching the feature cap raises an exception rather than being
recorded. Cap saturation silently injected fake error mass during Day 1 and would otherwise have
produced a spurious category effect, since longer prompts are unevenly distributed across
categories.

---

## 3. Outcomes

**Primary outcome:** replacement score (library implementation).

**Secondary outcomes:** completeness; total error mass; `error_by_layer` (26 values, comparable
across all prompts); `error_by_position` (length-varying, reduced to scalars — see 6.3); mean path
length; node and edge counts; pruning curve at thresholds 0.95 / 0.9 / 0.8 / 0.7.

**Predictors, all computable in milliseconds without running attribution:** category; token count;
next-token entropy; top-1 probability; mean token log-frequency (out-of-distribution proxy).

The predictors must be cheap. A triage rule requiring the attribution first is useless.

---

## 4. Predictions

### P1 — Category ranking (blind)

Predicted rank order by replacement score, best to worst:

1. Factual recall
2. Syntactic agreement
3. Two-digit arithmetic
4. Known vs unknown entity
5. Multi-hop
6. Code completion
7. Multiple choice
8. Induction / copying
9. Obfuscated text

Scored by Spearman correlation between predicted and observed ranks.

### P2 — Replacement score has low between-category variance (data-informed)

**I predict replacement score will not discriminate between categories.** Specifically: the
between-category range of category means will be **under 0.15**, and category will explain
**under 20%** of variance in replacement score.

*This is informed by pilot data.* Ten prompts spanning factual recall, arithmetic, syntax,
multi-hop, code and induction produced a replacement range of 0.684–0.755 — a spread of 0.071
against a noise floor of 5e-05. The differences are real but small. I am extrapolating that
tightness to the full corpus, including three categories the pilot did not cover (multiple choice,
known/unknown entity, obfuscated text).

**The extrapolation is the risky part.** Obfuscated text in particular is the case where the
methods paper reports near-total error-node dominance, so it could break the pattern badly. If it
does, P2 fails and that is the more interesting outcome.

### P3 — Error location has high between-category variance (data-informed direction, blind magnitude)

**I predict error *location* will discriminate where error *quantity* does not.** Specifically,
category will explain **more variance in error-location summaries than in replacement score**,
by a factor of at least 2.

Mechanistically: obfuscated text should concentrate error on corrupted tokens in early layers;
induction may concentrate at the copy-source position; factual recall should be low and flat.

### P4 — Completeness is even less discriminative than replacement

Pilot completeness ranged 0.925–0.942, a spread of 0.017 against replacement's 0.071. I predict
completeness will have a smaller between-category range than replacement across the full corpus.

### P5 — Human ratings (blind)

Spearman correlation between replacement score and my 3-point "did I learn a mechanism" rating on
40 graphs will be **positive but weak: |rho| < 0.4**.

Rationale: if P2 holds and replacement barely varies, it cannot track something as coarse as
whether a human understood the graph.

---

## 5. Falsification

The project has a null result if:

- Category explains **under 10%** of variance in every outcome, and no predictor reaches
  significance after Benjamini-Hochberg correction. Then no triage rule exists and I say so.
- Leave-one-category-out cross-validated R² is **at or below zero** for all outcomes. Then any
  in-sample structure does not generalise to unseen prompt types, which is the actual use case.

**I will report a null result as the headline finding rather than reframing around a surviving
sub-analysis.** A documented "these metrics do not support triage" is worth more to the field than
a fished positive.

Specific ways I could be wrong, stated now:

- If P2 fails and replacement *does* discriminate strongly, my central claim inverts and the metric
  is more useful than I expect. Good outcome, different paper.
- If P3 fails and error location is also flat, the project has no signal in either variable and
  the honest conclusion is that per-prompt triage is not achievable with these instruments on this
  model.
- If human ratings are self-inconsistent (intra-rater agreement below 0.6), P5 is uninterpretable
  and I report that instead of the correlation.

---

## 6. Analysis plan

### 6.1 Models

Outcomes are proportions bounded in (0,1). Beta regression, or logit-transform then OLS. Not plain
linear regression on a proportion.

- Outcome ~ category + token_count + next_token_entropy + top1_prob + mean_token_logfreq
- Effect sizes with confidence intervals, not p-values alone
- Benjamini-Hochberg across the 9 category contrasts and across outcomes

### 6.2 Cross-validation

Two schemes, both reported:

- **Held-out prompts within known categories.** The easy version.
- **Leave-one-category-out.** The version that matters, because the real use case is a prompt type
  I never tested. I expect this to be substantially worse and will say so plainly.

### 6.3 Handling variable-length error profiles

`error_by_position` has one value per token, so lengths differ across the corpus and the arrays
cannot be stacked directly. Pre-specified reduction to scalars:

- fraction of error mass in the first third / middle third / final third of positions
- position index of maximum error, normalised to [0, 1]
- Gini coefficient of the position profile (concentration vs spread)
- error mass at the second-to-last position

`error_by_layer` is always 26 values and needs no reduction. Summaries: fraction in the first
third / last third of layers, and layer index of maximum error.

**Note:** error mass at the *final* token position is structurally 0.000 once the feature cap is
not binding, so it is excluded as an outcome. This is itself a limitation of the instrument and
will be reported: the method's blind spots at the read-out position are not measurable this way.

### 6.4 Noise floor

Any claimed effect must exceed 5e-05 by a wide margin. Observed pilot spread (0.071) is ~1400x the
floor, so real effects are detectable; the question is whether they are *large*, not whether they
are *nonzero*.

### 6.5 Human validation

40 graphs sampled to span the observed range of replacement score, not stratified by category —
spread on the validated variable is what matters. Rated blind to all metrics, randomised order,
5-minute cap per graph, 3-point scale. 10 unmarked duplicates for intra-rater reliability, reported
regardless of outcome. Single-rater and non-blind-to-hypothesis; both stated as limitations.

---

## 7. Scope limits, stated in advance

- One model, one size, one dictionary family. Everything may differ at 9B or with cross-layer
  transcoders.
- bf16, not fp32. T4, not A100.
- Metrics measure whether a graph **covers** the computation, not whether it covers it
  **correctly**. Mechanistic faithfulness requires per-graph intervention experiments and is out
  of scope. This is the deepest limitation and will be stated prominently.
- Prompt categories are hand-defined by me and not exhaustive.
- I formed P2–P4 after seeing 10 pilot prompts.

---

## 8. Deliverables, committed regardless of outcome

1. Prompt corpus with per-prompt metrics, released on HuggingFace
2. The regression, with honest cross-validated performance
3. A one-page triage card, or an explicit statement that no usable rule was found
4. Write-up with epistemic status up front, posted to LessWrong / Alignment Forum

---

## 9. Deviations log

Append dated entries. Do not edit anything above.

| Date | Deviation | Reason |
|---|---|---|
| | | |

CHOICE = errors_first, confirmed against the library's 0.7196\
RESHAPE = layer_major, confirmed by the mod-6 zero pattern (26/26 vs flat)\
max_feature_nodes=8192 saturates above ~10 tokens and injects fake error mass at the final position; 16384 resolves it\
Final-position error is structurally 0.000 when the cap isn't binding — a real limitation of the instrument\
Noise floor 5e-05; structure exactly reproducible\
~13s per graph; 300 graphs ≈ 1.6h with margin\
P100 is sm_60 and unusable; T4 x2 only\
Weights are 17.2 GB and legitimately so — 3 GB free, so metrics-only saving on Day 3

CHOICE = errors_first
The error nodes and the token-embedding nodes sit next to each other in the adjacency matrix, in one unbroken block. Nothing about the structure says which group comes first. I tried both orderings and compared my replacement score against the library's own (0.7196); errors-first matched, embeddings-first would have given the inverted value. Getting this backwards would have reported 0.27 as 0.73 throughout.

RESHAPE = layer_major
The error nodes arrive as one flat list of numbers that has to be folded into a (layer × token) grid, and the folding order isn't documented. 56 of the values were exactly zero, and those zeros fell into two buckets of 26 under mod 6, versus a flat spread under mod 26. Two buckets of exactly 26 — one per layer — means the list is stored layer-by-layer. Structural evidence, not a judgement call.

max_feature_nodes: 8192 → 16384
The cap on how many feature nodes a graph may contain. Above about 10 tokens, graphs were hitting it, and the truncated features' contribution got dumped into error nodes — including at the final token position, which otherwise has none. That produced an error signal that looked like a property of the prompt but was really "this prompt hit the ceiling." Since longer prompts aren't evenly spread across categories, it would have manufactured a fake category effect. Raising to 16384 stops the ceiling binding.

Final-position error is structurally zero
Once the cap isn't binding, the last token position has no error nodes at all — the method appears to fold that position's reconstruction gap into the output rather than representing it separately. So the graph cannot tell you whether it went blind at the position the model actually reads from. That's a gap in the instrument, not in my code, and it belongs in the limitations.

Noise floor 5e-05
Running identical prompts twice gives replacement scores differing by about 0.00005. That's bf16 arithmetic — the GPU picks reduction orders at runtime and low-precision rounding doesn't cancel. Node counts match exactly, so the graph structure is deterministic and only the arithmetic wobbles. Any effect I claim later has to be much larger than this.

~13s per graph
Measured on ten prompts of varying length at the final settings. 300 graphs is about 1.1 hours, 1.6 with a safety margin, against a 10-hour budget. Comfortable surplus, which means the corpus stays at 300 and there's room for the cross-layer transcoder comparison.

P100 unusable
Kaggle offers two accelerators. The P100 is an older architecture (compute capability 6.0) and the installed PyTorch only ships compiled kernels for 7.0 and above, so it fails with "no kernel image available." Not fixable by tuning. T4 x2 for everything.

Weights are 17.2 GB
Gemma-2-2B is 9.8 GB and the GemmaScope transcoders 7.4 GB — both as published, and the bf16 cast happens after download. No duplication to clear. That leaves 3 GB of the 20 GB quota, and a single graph saves at 685 MB. So Day 3 records metrics only and discards each graph; the 40 needed for hand-rating get re-run on Day 5 at nine minutes total.

In [ ]:
import json
print(json.dumps(json.load(open("/kaggle/working/out/day1_config.json")), indent=2))

# Day 1 (v4) — setup, metrics, timing

Changes from v3, all driven by what the v3 run found:

| Finding | Change |
|---|---|
| `batch_size=64` peaked at 14.3 GB on a 14.56 GB card; 32 peaked at 6.5 GB and was no slower | `batch_size=32` fixed in `ATTR_KWARGS` |
| OOM traced to Gemma-2's logit soft-capping over a 256k vocab, not feature count | Prompt length ceiling probe added |
| `err_fin = 0.000` on 7 of 8 prompts — implausible | Part 6 rewritten with an **oracle test** that settles the reshape empirically |
| Determinism check failed at `rtol=1e-9` on 1e-4 differences | Tolerance loosened to 1e-3; run-to-run noise recorded as a floor |
| Pilot failures truncated to 130 chars, hiding the real error | Full tracebacks |
| Disk down to 3 GB | Disk guard, and a policy for which `.pt` files to keep |

Accelerator: **GPU T4 x2**. Backend: nnsight (set `BACKEND` below if you switch).

---

## Part 0 — Install, then restart

Run alone, then **Run → Restart kernel**, then continue from Part 1.

In [ ]:
!pip install -q circuit-tracer
print("installed — now Run > Restart kernel, then continue from Part 1")

## Part 1 — Environment, hardware, auth

In [ ]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
cap = torch.cuda.get_device_capability(0)
GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU: {GPU_NAME}  sm_{cap[0]}{cap[1]}")
assert cap >= (7, 0), (
    f"{GPU_NAME} is sm_{cap[0]}{cap[1]}; this PyTorch supports sm_70+. "
    "Settings > Accelerator > GPU T4 x2, then restart."
)
print("hardware ok")

In [ ]:
import time, gc

def hw(peak=False):
    ram = os.popen("free -g | awk 'NR==2{print $7}'").read().strip() or "?"
    disk = os.popen("df -BG /kaggle/working | awk 'NR==2{print $4}'").read().strip()
    a = torch.cuda.memory_allocated() / 1e9
    t = torch.cuda.get_device_properties(0).total_memory / 1e9
    s = f"RAM free {ram} GB | GPU {a:.1f}/{t:.1f} GB | disk free {disk}"
    if peak:
        s += f" | GPU peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB"
    print(s)

def free():
    gc.collect(); torch.cuda.empty_cache()

hw()
# Three separate resources, all of which have caused failures:
#   system RAM  -> silent kernel restart during model load
#   GPU memory  -> OOM inside Gemma-2's logit soft-capping
#   disk        -> 20 GB shared between the HF cache and saved graphs

In [ ]:
from kaggle_secrets import UserSecretsClient
_tok = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = _tok
os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok

from huggingface_hub import login, whoami
login(token=_tok)
print("logged in as:", whoami()["name"])

for d in ("graphs", "out"):
    os.makedirs(f"/kaggle/working/{d}", exist_ok=True)

In [ ]:
# Ask pip, not the module: transformer_lens has no __version__.
# Distribution names use hyphens; import names use underscores.
from importlib.metadata import version, PackageNotFoundError

def ver(dist):
    try:
        return version(dist)
    except PackageNotFoundError:
        return "unknown"

VERSIONS = {"torch": torch.__version__, "transformers": ver("transformers"),
            "transformer_lens": ver("transformer-lens"), "circuit_tracer": ver("circuit-tracer"),
            "nnsight": ver("nnsight"), "numpy": ver("numpy"), "scipy": ver("scipy"),
            "gpu": GPU_NAME}
for k, v in VERSIONS.items():
    print(f"{k:20s} {v}")

## Part 2 — Load

nnsight skips TransformerLens's state-dict conversion, which was killing the kernel on system RAM.
No `hf_model` pre-load needed on this path — that cell only helps TransformerLens, and holding an
extra 5 GB of HF model in RAM would work against you here.

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("google/gemma-2-2b")
print("weights cached"); hw()

In [ ]:
from circuit_tracer import ReplacementModel, attribute, Graph

BACKEND = "nnsight"
LOAD_KWARGS = dict(dtype=torch.bfloat16, device="cuda", lazy_encoder=True)

t0 = time.time()
model = ReplacementModel.from_pretrained(
    "google/gemma-2-2b", "gemma", backend=BACKEND, **LOAD_KWARGS
)
print(f"loaded in {time.time()-t0:.0f}s")
free(); hw()

# bf16 not fp32: fp32 needs ~14.3 GB for model + transcoders and fits no Kaggle GPU.
# bf16 not fp16: Gemma-2 has logit soft-capping and large activations; fp16 overflows.
# lazy_encoder=True: encoders are the resident half (lazy_decoder defaults to True).

### TransformerLens alternative

If you switch back, the two-stage load avoids the RAM kill — build the HF model yourself in bf16
with `low_cpu_mem_usage=True`, then pass it in:

```python
from transformers import AutoModelForCausalLM
hf = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-2b", torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
model = ReplacementModel.from_pretrained(
    "google/gemma-2-2b", "gemma", hf_model=hf, **LOAD_KWARGS)
del hf; free()
```

## Part 3 — Settings and first graph

In [ ]:
import inspect
print("attribute", inspect.signature(attribute))
# Installed signature is attribute(prompt, model, ...), NOT the documented attribute(model, prompt).

In [ ]:
# batch_size=32 from the v3 sweep: peak 6.5 GB vs 14.3 GB at 64, and marginally faster.
# 64 left 1.8% headroom on a 14.56 GB card, which would fail intermittently in an unattended run.

# Changed to 16384 after discovering lower limit affected err_fin -> 0 (for graphs that saturate the nodes)?
ATTR_KWARGS = dict(max_feature_nodes=16384, batch_size=32)

PROMPT = "The capital of France is"

t0 = time.time(); g = attribute(prompt=PROMPT, model=model, **ATTR_KWARGS)
print(f"run 1 (warm-up included): {time.time()-t0:.1f}s")

torch.cuda.reset_peak_memory_stats()
t0 = time.time(); g = attribute(prompt=PROMPT, model=model, **ATTR_KWARGS)
T_WARM = time.time() - t0
print(f"run 2 (the real number):  {T_WARM:.1f}s")
hw(peak=True)

In [ ]:
ATTR_KWARGS

In [ ]:
# Prompt length ceiling. The v3 OOM happened inside
#   modeling_gemma2.py: logits = logits / self.config.final_logit_softcapping
# which is elementwise over batch x seq_len x 256128. So the binding constraint is sequence
# length through a vocab-sized tensor, not feature count. Find the ceiling at batch_size=32.
def probe_len(n_tokens):
    p = "The quick brown fox jumps over the lazy dog and then runs far away today " * 3
    ids = model.tokenizer(p)["input_ids"][:n_tokens]
    txt = model.tokenizer.decode(ids)
    torch.cuda.reset_peak_memory_stats()
    try:
        t0 = time.time()
        gp = attribute(prompt=txt, model=model, **ATTR_KWARGS)
        print(f"{n_tokens:3d} tokens: ok  {time.time()-t0:5.1f}s  "
              f"peak {torch.cuda.max_memory_allocated()/1e9:4.1f} GB  nodes {gp.adjacency_matrix.shape[0]}")
        del gp; free(); return True
    except Exception as e:
        print(f"{n_tokens:3d} tokens: FAILED {type(e).__name__}")
        free(); return False

for n in (12, 16, 20, 24):
    if not probe_len(n):
        break

print("\nSet MAX_TOKENS below the first failure and apply it to ALL nine categories,")
print("so prompt length is not confounded with category.")

In [ ]:
MAX_TOKENS = 24   # <-- set from the probe above

## Part 4 — The library's own metrics (reference)

In [ ]:
import circuit_tracer.graph as ctg
print("callables in circuit_tracer.graph:")
for n in dir(ctg):
    if not n.startswith("_") and callable(getattr(ctg, n)):
        print("  ", n)

In [ ]:
METRIC_FN = None
for n in dir(ctg):
    if n.startswith("_"):
        continue
    o = getattr(ctg, n)
    if callable(o) and o.__doc__ and "eplacement" in o.__doc__ and "ompleteness" in o.__doc__:
        METRIC_FN = o
        print("found:", n, inspect.signature(o), "\n")
        print(inspect.getdoc(o)[:1500]); break

if METRIC_FN is None:
    import subprocess
    print(subprocess.run(["sed", "-n", "310,400p", ctg.__file__],
                         capture_output=True, text=True).stdout)

In [ ]:
LIB_SCORES = None
if METRIC_FN is not None:
    for call in (lambda: METRIC_FN(g), lambda: METRIC_FN(graph=g),
                 lambda: METRIC_FN(g.adjacency_matrix, g.logit_probabilities)):
        try:
            LIB_SCORES = call(); break
        except Exception as e:
            print("tried:", type(e).__name__, str(e)[:110])
print("\nlibrary scores:", LIB_SCORES, "  <-- ours must match these")

## Part 5 — Node partition

Node types are recovered from graph structure, not from index conventions:
error and embedding nodes have **no incoming edges**; logit nodes have **no outgoing edges**.

Two v3 corrections are baked in. Errors and embeddings sit in one **contiguous** run
(156 + 6 = 162 for a 6-token prompt), so they must be split by arithmetic rather than by
contiguity. And zero-out-degree caught 178 nodes, not 10 — dead feature nodes have no outgoing
edges either — so logits are narrowed by incoming weight.

In [ ]:
import numpy as np, scipy.sparse as sp

def to_np(x):
    return x.detach().cpu().float().numpy() if torch.is_tensor(x) else np.asarray(x)

def contiguous_blocks(idx):
    if len(idx) == 0:
        return []
    out, s, p = [], idx[0], idx[0]
    for i in idx[1:]:
        if i != p + 1:
            out.append((s, p)); s = i
        p = i
    out.append((s, p)); return out


def find_nodes(A0, n_tok, n_layers, n_log):
    """Return (err_first, emb_first, logits) index arrays, derived from structure."""
    nz = np.abs(A0) > 0
    zin = np.where(nz.sum(axis=1) == 0)[0]     # no incoming: errors + embeddings
    zout = np.where(nz.sum(axis=0) == 0)[0]    # no outgoing: logits + dead features

    run = max(contiguous_blocks(zin), key=lambda b: b[1] - b[0])
    span = run[1] - run[0] + 1
    if span != n_layers * n_tok + n_tok:
        raise ValueError(f"zero-in run is {span}, expected {n_layers*n_tok + n_tok}")

    ne = n_layers * n_tok
    err_first = (np.arange(run[0], run[0] + ne), np.arange(run[0] + ne, run[1] + 1))
    emb_first = (np.arange(run[0] + n_tok, run[1] + 1), np.arange(run[0], run[0] + n_tok))

    lg = np.setdiff1d(zout, zin)
    if len(lg) != n_log:
        lg = lg[np.argsort(-np.abs(A0[lg]).sum(axis=1))[:n_log]]
    return err_first, emb_first, lg


N_LAYERS = model.cfg.n_layers
A_raw = to_np(g.adjacency_matrix).astype(np.float64)
N_TOK = len(to_np(g.input_tokens))
N_LOG = len(to_np(g.logit_probabilities))
print(f"nodes={A_raw.shape[0]} tokens={N_TOK} layers={N_LAYERS} logits={N_LOG}")
print(f"expected errors {N_LAYERS}x{N_TOK}={N_LAYERS*N_TOK}, embeddings {N_TOK}")

ef, mf, LOG = find_nodes(A_raw, N_TOK, N_LAYERS, N_LOG)
print(f"logits identified: {len(LOG)}")

In [ ]:
def influence(A_raw, logit_idx, logit_probs, absorb_idx, max_iter=500):
    """Probability-weighted total path strength from every node to the logits.

    Normalise incoming edges to sum to 1, then push mass backwards from the logits one edge at
    a time. The graph is a DAG so the series terminates exactly. Also returns the path-length
    distribution of mass absorbed at `absorb_idx`.
    """
    A = np.abs(A_raw)
    rs = A.sum(axis=1, keepdims=True)
    A = np.divide(A, rs, out=np.zeros_like(A), where=rs > 0)
    AT = sp.csr_matrix(A.T)
    v = np.zeros(A.shape[0]); v[logit_idx] = logit_probs
    infl = np.zeros(A.shape[0]); by_len = []
    for k in range(1, max_iter + 1):
        v = AT @ v
        infl += v
        by_len.append(v[absorb_idx].sum())
        if v.sum() < 1e-12:
            break
    return A, infl, np.array(by_len), k


lp = to_np(g.logit_probabilities); lp = lp / lp.sum()

print("split          replacement   error     sum")
res = {}
for label, (er, em) in (("errors_first", ef), ("embeds_first", mf)):
    A, infl, by_len, steps = influence(A_raw, LOG, lp, em)
    r, e = float(infl[em].sum()), float(infl[er].sum())
    res[label] = (r, e, A, infl, by_len, steps, er, em)
    print(f"{label:14s}  {r:.4f}      {e:.4f}   {r+e:.6f}")

print("\nConservation holds for BOTH splits — all zero-in-degree mass is absorbed either way.")
print("It proves the run is complete, not which half is which.")
print(f"The library's score decides: {LIB_SCORES}")

In [ ]:
CHOICE = "errors_first"   # <-- set to whichever matches LIB_SCORES

r, e, A, infl, by_len, steps, ERR, EMB = res[CHOICE]
assert steps < 500, "series did not terminate — cycle, i.e. wrong orientation"
assert abs(r + e - 1.0) < 1e-4, f"conservation violated by {abs(r+e-1):.2e}"

em = np.zeros(A.shape[0], bool); em[ERR] = True
w = infl * (A.sum(axis=1) > 0)
comp = float(1 - (w @ A[:, em].sum(axis=1)) / w.sum())
mpl = float((np.arange(1, len(by_len) + 1) @ by_len) / by_len.sum())

print(f"replacement  {r:.4f}   (published CLT 0.61, PLT 0.37)")
print(f"completeness {comp:.4f}   (published CLT 0.80, PLT 0.78)")
print(f"path length  {mpl:.2f}     (published CLT 2.3, PLT 3.7)")
assert comp > r, "completeness must exceed replacement — check path accounting"

## Part 6 — Error decomposition, and settling the reshape

**Why 156.** An MLP runs once per layer *per token position*, so the transcoder's reconstruction
gap is measured separately at every (layer, position) pair: 26 x 6 = 156 error nodes. Change the
prompt length and the count changes with it.

**What we measure.** `influence[ERR]` reshaped to a (layer, position) grid says *where* the method
went blind — not just how much was unexplained, but at which token and which depth. Influence
rather than raw magnitude matters: an error node the output never depends on is not a blind spot
you care about.

**The v3 bug.** `err_fin = 0.000` on 7 of 8 prompts, while total error was ~0.28. The final token
is what the logits read from, so zero error there is not plausible — the grid was scrambled.
NumPy fills row by row, so `reshape(26, 6)` assumes layer-then-position storage while
`reshape(6, 26).T` assumes the reverse. Both fit 156 numbers and both give the same total, so only
the *profile* distinguishes them.

Three tests below, in increasing order of authority.

In [ ]:
# TEST 2 — the oracle. Put a nonsense token at a KNOWN position. The transcoder is trained on
# natural text, so an off-distribution token should blow up reconstruction error right there.
# Whichever reshape localises the spike at that index is the correct one.
ORACLE = "The capital of Fzzqxw is the city"
oracle_pos = None
ids = model.tokenizer(ORACLE)["input_ids"]
otoks = [model.tokenizer.decode([t]) for t in ids]
print("tokens:", list(enumerate(otoks)))

go = attribute(prompt=ORACLE, model=model, **ATTR_KWARGS)
Ao = to_np(go.adjacency_matrix).astype(np.float64)
nto = len(to_np(go.input_tokens))
efo, mfo, lgo = find_nodes(Ao, nto, N_LAYERS, len(to_np(go.logit_probabilities)))
ero, embo = (efo if CHOICE == "errors_first" else mfo)
lpo = to_np(go.logit_probabilities); lpo = lpo / lpo.sum()
_, infl_o, _, _ = influence(Ao, lgo, lpo, embo)

lm = infl_o[ero].reshape(N_LAYERS, nto).sum(axis=0)
pm = infl_o[ero].reshape(nto, N_LAYERS).T.sum(axis=0)

import pandas as pd
print(pd.DataFrame({"pos": range(nto), "token": otoks[:nto],
                    "layer_major": np.round(lm, 4),
                    "position_major": np.round(pm, 4)}).to_string(index=False))
print("\nThe nonsense token's index should carry a visible spike in the CORRECT column.")
del go; free()

In [ ]:
# TEST 1 — read the convention out of the source. create_graph_files has to know the node
# layout in order to emit the frontend JSON.
import subprocess
from circuit_tracer.utils import create_graph_files


path = create_graph_files.__globals__["__file__"]


pat = "|".join(["error", "embed", "logit", "n_pos", "reshape", "ctx"])
out = subprocess.run(["grep", "-n", "-i", "-E", pat, path],
                     capture_output=True, text=True).stdout
print(out[:3000])

In [ ]:
# TEST 3 — the final-position check, on the ordinary prompt. Late-layer error at the last
# token feeds the logits directly, so a zero there is a red flag.
toks = [model.tokenizer.decode([t]) for t in to_np(g.input_tokens).astype(int)]
lm2 = infl[ERR].reshape(N_LAYERS, N_TOK)
pm2 = infl[ERR].reshape(N_TOK, N_LAYERS).T

print(pd.DataFrame({"pos": range(N_TOK), "token": toks,
                    "layer_major": np.round(lm2.sum(axis=0), 4),
                    "position_major": np.round(pm2.sum(axis=0), 4)}).to_string(index=False))
print(f"\ntotals identical by construction: {lm2.sum():.4f} vs {pm2.sum():.4f}")
print("Reshaping never changes the sum, so the total is not diagnostic. The profile is.")
print("\nAlso check the layer profiles — the correct one should show structure with depth,")
print("not noise:")
print("  layer-major   :", np.round(lm2.sum(axis=1), 3))
print("  position-major:", np.round(pm2.sum(axis=1), 3))

In [ ]:
raw = infl[ERR]
z = np.where(raw == 0)[0]
print(f"{len(z)} exact zeros of {len(raw)}")
print("zeros mod 6 :", np.bincount(z % 6, minlength=6))
print("zeros mod 26:", np.bincount(z % 26, minlength=26))

In [ ]:
RESHAPE = "layer_major"   # <-- set from tests 1-3 above

grid = lm2 if RESHAPE == "layer_major" else pm2
print("error by position:", np.round(grid.sum(axis=0), 4))
print("error by layer   :", np.round(grid.sum(axis=1), 4))
print(f"final position   : {grid.sum(axis=0)[-1]:.4f}")
# err_fin is structurally zero on this model: the final position's error contribution is not
# represented as a separate error node. Confirmed by the mod-6 zero pattern (Part 6, Test A).
print(f"final position error: {grid.sum(axis=0)[-1]:.4f}")

## Part 7 — Pruning (library implementation)

In [ ]:
print(inspect.signature(ctg.prune_graph))
print(inspect.getdoc(ctg.prune_graph)[:900])

In [ ]:
def pruning_curve(graph, thresholds=(0.95, 0.9, 0.8, 0.7)):
    out = {}
    for t in thresholds:
        try:
            res_ = ctg.prune_graph(graph, node_threshold=t, edge_threshold=t)
            mask = getattr(res_, "node_mask", None)
            out[str(t)] = int(mask.sum()) if mask is not None else None
        except Exception as ex:
            out[str(t)] = f"ERR {type(ex).__name__}"
    return out

print(pruning_curve(g))
print("\nThe library's version does iterative removal of orphaned nodes; an influence-rank cut")
print("does not, and the difference shows up most on exactly the messy graphs we care about.")

## Part 8 — Assemble

In [ ]:
CONFIG = dict(
    model="google/gemma-2-2b", transcoders="gemma", backend=BACKEND,
    dtype="bfloat16", device="cuda", lazy_encoder=True,
    n_layers=N_LAYERS, split=CHOICE, reshape=RESHAPE,
    attr_kwargs=ATTR_KWARGS, max_tokens=MAX_TOKENS,
    tolerance=1e-4, versions=VERSIONS,
)


def graph_metrics(gr, cfg=CONFIG):
    A0 = to_np(gr.adjacency_matrix).astype(np.float64)
    nt = len(to_np(gr.input_tokens))
    nl, nlg = cfg["n_layers"], len(to_np(gr.logit_probabilities))

    ef_, mf_, lg = find_nodes(A0, nt, nl, nlg)
    er, emb = ef_ if cfg["split"] == "errors_first" else mf_

    lw = to_np(gr.logit_probabilities); lw = lw / lw.sum()
    Am, inf, bl, st = influence(A0, lg, lw, emb)

    rr, ee = float(inf[emb].sum()), float(inf[er].sum())
    if abs(rr + ee - 1.0) > cfg["tolerance"]:
        raise ValueError(f"conservation violated: {rr+ee}")

    mk = np.zeros(Am.shape[0], bool); mk[er] = True
    ww = inf * (Am.sum(axis=1) > 0)
    gd = (inf[er].reshape(nl, nt) if cfg["reshape"] == "layer_major"
          else inf[er].reshape(nt, nl).T)

    return {
        "replacement_score": rr,
        "completeness_score": float(1 - (ww @ Am[:, mk].sum(axis=1)) / ww.sum()),
        "error_mass_total": ee,
        # "error_mass_final_pos": float(gd.sum(axis=0)[-1]),
        "error_by_position": gd.sum(axis=0).tolist(),
        "error_by_layer": gd.sum(axis=1).tolist(),
        "mean_path_length": float((np.arange(1, len(bl) + 1) @ bl) / bl.sum()),
        "n_nodes": int(Am.shape[0]), "n_edges": int((A0 != 0).sum()),
        "n_tokens": nt, "series_steps": st,
        "pruning_nodes": pruning_curve(gr),
    }


m = graph_metrics(g)
for k, v in m.items():
    if not isinstance(v, (list, dict)):
        print(f"{k:24s} {v}")

## Part 9 — Noise floor and timing pilot

Run-to-run differences of ~1e-4 are bf16 rounding, not bugs: cuBLAS picks reduction orders at
runtime and low-precision rounding doesn't cancel. `n_nodes` matching exactly is the meaningful
part — structure is deterministic, arithmetic wobbles. Record the spread as a floor; any effect
claimed on Day 6 has to be much larger than it.

In [ ]:
g2 = attribute(prompt=PROMPT, model=model, **ATTR_KWARGS)
m2 = graph_metrics(g2)

NOISE = {}
for k in ("replacement_score", "completeness_score", "mean_path_length"):
    d = abs(m[k] - m2[k]); NOISE[k] = d
    print(f"{k:22s} {m[k]:.6f} vs {m2[k]:.6f}  delta {d:.2e}  "
          f"{'ok' if np.isclose(m[k], m2[k], rtol=1e-3) else 'DIFFERS'}")
print(f"n_nodes                {m['n_nodes']} vs {m2['n_nodes']}  "
      f"{'ok' if m['n_nodes']==m2['n_nodes'] else 'DIFFERS'}")

CONFIG["noise_floor"] = NOISE
del g2; free()

In [ ]:
import traceback

PILOT = [
    "The capital of France is",
    "2 + 3 =",
    "The cat sat on the",
    "Michael Jordan plays the sport of",
    "calc: 36+59=",
    "The largest planet in our solar system is called",
    "def add(a, b):\n    return",
    "The blorp ran fast. The blorp",                      # induction, short
    "The capital of the state containing Dallas is",
    "In 1969 the first humans on the moon flew mission",   # shortened to fit MAX_TOKENS
]

rows = []
for pr in PILOT:
    n_tok = len(model.tokenizer(pr)["input_ids"])
    if n_tok > MAX_TOKENS:
        print(f"SKIP ({n_tok} > {MAX_TOKENS} tokens): {pr[:40]}")
        continue
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        gp = attribute(prompt=pr, model=model, **ATTR_KWARGS)
        dt = time.time() - t0
        mm = graph_metrics(gp)
        rows.append(dict(prompt=pr[:36], secs=round(dt, 1), n_tok=mm["n_tokens"],
                         rep=round(mm["replacement_score"], 3),
                         comp=round(mm["completeness_score"], 3),
                         # err_fin=round(mm["error_mass_final_pos"], 3),
                         nodes=mm["n_nodes"],
                         peak_gb=round(torch.cuda.max_memory_allocated()/1e9, 1)))
        del gp
    except Exception:
        rows.append(dict(prompt=pr[:36], secs=round(time.time()-t0, 1), n_tok=n_tok,
                         rep=None, comp=None, err_fin=None, nodes=None, peak_gb=None))
        print(f"\nFAILED: {pr[:40]}")
        traceback.print_exc()
    free()

pilot = pd.DataFrame(rows)
print(pilot.to_string(index=False))
hw()

In [ ]:
ok = pilot.dropna(subset=["rep"])
per = ok["secs"].mean()
print(f"{len(ok)}/{len(pilot)} succeeded | mean {per:.1f}s | max {ok['secs'].max():.1f}s "
      f"| peak GPU {ok['peak_gb'].max():.1f} GB")
print(f"300 graphs: {300*per/3600:.1f} h | x1.5 margin: {1.5*300*per/3600:.1f} h  (budget 10 h)")
print()
print("err_fin non-zero on:", int((ok['err_fin'] > 0).sum()), "of", len(ok))
print("If most are still zero, the RESHAPE switch is wrong — fix before Day 2.")
print()
print("replacement spread:", round(ok['rep'].min(), 3), "to", round(ok['rep'].max(), 3),
      "| noise floor", f"{CONFIG['noise_floor']['replacement_score']:.1e}")
print("A tight spread means replacement score cannot be the triage signal on its own,")
print("and the weight of the project shifts to error LOCATION rather than error QUANTITY.")

## Check Storeage sopace

In [ ]:
g_big = attribute(prompt="In 1969 the first humans on the moon flew mission",
                  model=model, **ATTR_KWARGS)
g_big.to_pt("/tmp/test.pt")
print(f"{os.path.getsize('/tmp/test.pt')/1e6:.0f} MB")
!rm /tmp/test.pt
!du -sh /kaggle/working/hf

In [ ]:
!du -sh /kaggle/working/hf/hub/* | sort -h

### Additions : test whether the err_fin is always 0 and does it differ from semantic (and, but, the) and content words (called, return, blorp, mission)

In [ ]:
for pr in ["calc: 36+59=", "The largest planet in our solar system is called"]:
    print(pr, "->", [model.tokenizer.decode([t]) for t in model.tokenizer(pr)["input_ids"]])

In [ ]:
PAIRS = [
    # 8 tokens each
    ("The doctor examined the patient and then left",      "content"),
    ("The doctor examined the patient and looked at",      "function"),
    # 11 tokens each
    ("She opened the heavy wooden door and walked inside slowly",  "content"),
    ("She opened the heavy wooden door and walked over to the",    "function"),
    # 14 tokens each
    ("After the long meeting ended everyone went home to rest for the evening", "content"),
    ("After the long meeting ended everyone went home to rest and then talked about", "function"),
]

for p, kind in PAIRS:
    n = len(model.tokenizer(p)["input_ids"])
    gp = attribute(prompt=p, model=model, **ATTR_KWARGS)
    mm = graph_metrics(gp)
    print(f"{n:3d} tok  {kind:9s}  err_fin {mm['error_mass_final_pos']:.3f}  "
          f"rep {mm['replacement_score']:.3f}  nodes {mm['n_nodes']}  <- {p[-20:]!r}")
    del gp; free()

In [ ]:
P = "She opened the heavy wooden door and walked inside slowly"
for cap in (8192, 16384, 32768):
    gp = attribute(prompt=P, model=model, max_feature_nodes=cap, batch_size=32)
    mm = graph_metrics(gp)
    print(f"cap {cap:6d}  nodes {mm['n_nodes']:6d}  err_fin {mm['error_mass_final_pos']:.3f}  "
          f"rep {mm['replacement_score']:.3f}  comp {mm['completeness_score']:.3f}")
    del gp; free()

In [ ]:
LONG = "After the long meeting ended everyone went home to rest and then talked about"
gp = attribute(prompt=LONG, model=model, max_feature_nodes=16384, batch_size=32)
mm = graph_metrics(gp)
print(f"nodes {mm['n_nodes']}  err_fin {mm['error_mass_final_pos']:.4f}  "
      f"peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
del gp; free()

## Part 10 — Disk policy and freeze

A 4581-node graph in float64 is ~170 MB as `.pt`. At 300 graphs that is ~50 GB against a 20 GB
quota, shared with the ~5 GB HF cache. So: metrics for all 300, `.pt` files only for the 40 you
will hand-rate on Day 5.

In [ ]:
print("largest items under /kaggle/working:")
print(os.popen("du -sh /kaggle/working/* 2>/dev/null | sort -h | tail -8").read())
hw()

sz = os.path.getsize("/kaggle/working/graphs/france.pt") / 1e6 if \
     os.path.exists("/kaggle/working/graphs/france.pt") else 0
print(f"\none graph .pt = {sz:.0f} MB -> 300 graphs would be {sz*300/1000:.1f} GB")
print("Day 3 policy: save metrics rows always; save .pt only for the rating subsample.")

In [ ]:
import json
json.dump(m, open("/kaggle/working/out/reference_france.json", "w"), indent=2)
json.dump(CONFIG, open("/kaggle/working/out/day1_config.json", "w"), indent=2)
pilot.to_csv("/kaggle/working/out/day1_pilot.csv", index=False)
print("saved reference_france.json, day1_config.json, day1_pilot.csv")
print()
print("Day 3 regression test (atol=1e-3, not 1e-9 — bf16 noise):")
print("  ref = json.load(open('/kaggle/working/out/reference_france.json'))")
print("  assert np.isclose(graph_metrics(g)['replacement_score'], ref['replacement_score'], atol=1e-3)")

In [ ]:
import json, os
os.makedirs("/kaggle/working/out", exist_ok=True)

json.dump(m, open("/kaggle/working/out/reference_france.json", "w"), indent=2)
json.dump(CONFIG, open("/kaggle/working/out/day1_config.json", "w"), indent=2)
pilot.to_csv("/kaggle/working/out/day1_pilot.csv", index=False)

print(os.listdir("/kaggle/working/out"))
print(json.dumps(CONFIG, indent=2))

## Done when all seven hold

- [x] Conservation within 1e-4
- [x] Completeness > replacement, both plausible
- [x] Series terminated early
- [ ] Our numbers match the library's — this settles `CHOICE`
- [ ] ~~`err_fin` non-zero on most pilot prompts — this settles `RESHAPE`~~ RESHAPE confirmed by mod-6 zero pattern (26 in each of two buckets vs flat for mod 26)
- [ ] Noise floor recorded, structure (`n_nodes`) exactly reproducible
- [ ] Per-graph time, prompt-length ceiling, and a corpus-size decision

**Three switches set by hand**, each from the diagnostic printed above it: `MAX_TOKENS` (Part 3),
`CHOICE` (Part 5), `RESHAPE` (Part 6). All three are recorded in `CONFIG`.

**For the write-up:** these metrics measure whether the graph *covers* the computation, not whether
it covers it *correctly*. Mechanistic faithfulness needs per-graph interventions and is out of
scope. State bf16 not fp32, T4 not A100, `batch_size=32`, `lazy_encoder=True`, the backend, the
prompt-length ceiling, the noise floor, and `VERSIONS`.

PREREGISTRATION

The purpose of this section is to predefine the choices that we will make to perform the experiment, eliminating the temptation to tweak settings later and possibly improve the results. 

